# 2VA — Simulação C_alt: Modelo Único (30 seeds)

**Cenário H0:** Transfer Learning (XLM-RoBERTa) + Transformers, **sem** Ensemble  
Uma seed → um modelo treinado → métricas registradas  
Repete 30 vezes com seeds distintas → salva CSV para análise estatística

## Passo 1 — Setup

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from transformers import get_linear_schedule_with_warmup

# paths
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from src.models.classifier import HotelReviewClassifier
from src.models.trainer   import MultiTaskLoss, train_epoch, eval_epoch
from src.data.dataset     import ReviewDataset

# ── configuração da simulação ──────────────────────────────────────────────────
SEEDS        = list(range(30))       # seeds 0..29 — mesmas serão usadas no ensemble
N_EPOCHS     = 3                     # epochs por simulação (balanceia tempo × qualidade)
BATCH_SIZE   = 16
LR           = 2e-5
VAL_SPLIT    = 0.2                   # 80% treino / 20% validação

DATA_PATH    = project_root / "data" / "processed" / "reviews_labeled.csv"
RESULTS_PATH = Path("../results/single_model_results.csv")
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device : {DEVICE}")
print(f"seeds  : {SEEDS[0]}..{SEEDS[-1]}  ({len(SEEDS)} simulacoes)")
print(f"epochs : {N_EPOCHS} por simulacao")

## Passo 2 — Dados

O dataset é carregado **uma única vez** fora do loop de simulações.  
A cada seed, `random_split` embaralha e divide diferente — garantindo que cada modelo vê uma partição distinta.

In [ ]:
# carrega dataset completo uma única vez (tokenizer é pesado — não recarregar no loop)
full_dataset = ReviewDataset(str(DATA_PATH))

n_total = len(full_dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

print(f"total  : {n_total} reviews")
print(f"treino : {n_train}  |  validacao : {n_val}")

def make_loaders(seed: int):
    """Divide o dataset com a seed dada e retorna (train_loader, val_loader)."""
    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val], generator=generator)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader

## Passo 3 — `run_single_model(seed)`

Uma seed entra → um modelo treinado sai → F1 macro e MAE registrados.

In [ ]:
def set_seed(seed: int):
    """Fixa todas as fontes de aleatoriedade para garantir reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_single_model(seed: int) -> dict:
    set_seed(seed)

    # dados com split determinístico para esta seed
    train_loader, val_loader = make_loaders(seed)

    # modelo parte sempre dos pesos pré-treinados do XLM-RoBERTa (Transfer Learning)
    model     = HotelReviewClassifier().to(DEVICE)
    criterion = MultiTaskLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    total_steps = N_EPOCHS * len(train_loader)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    # loop de treino
    for epoch in range(N_EPOCHS):
        train_epoch(model, train_loader, optimizer, criterion, DEVICE, scheduler)

    # avaliação final na partição de validação
    metrics = eval_epoch(model, val_loader, DEVICE)

    return {
        "seed":     seed,
        "f1_macro": round(metrics["f1_macro"], 4),
        "mae":      round(metrics["mae"], 4),
    }

## Passo 4 — Loop das 30 simulações

In [ ]:
results = []

for i, seed in enumerate(SEEDS):
    print(f"[{i+1:02d}/30] seed={seed} ...", end=" ", flush=True)
    row = run_single_model(seed)
    results.append(row)
    print(f"F1={row['f1_macro']:.4f}  MAE={row['mae']:.4f}")

print("\nSimulacoes concluidas.")

## Passo 5 — Salvar CSV + preview

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULTS_PATH, index=False)
print(f"Salvo em: {RESULTS_PATH}")

# resumo estatístico — o que vai entrar no relatório
summary = df[["f1_macro", "mae"]].agg(["mean", "std", "min", "max"]).round(4)
print("\n--- Resumo das 30 simulacoes (C_alt: modelo unico) ---")
print(summary.to_string())

df